In [1]:
import sys
sys.path.append('../..')
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
import numpy as np

## Load the dataset

In [2]:
import ampligraph
# Benchmark datasets are under ampligraph.datasets module
from ampligraph.datasets import load_fb15k_237
# load fb15k-237 dataset
dataset = load_fb15k_237(return_mapper=True)

## Train the model

In [3]:
# Import the KGE model
from ampligraph.latent_features import ScoringBasedEmbeddingModel

# you can continue training from where you left after restoring the model
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir='./transe_train_logs')

# create the model with transe scoring function
model = ScoringBasedEmbeddingModel(eta=5, 
                                     k=300,
                                     scoring_type='TransE')

# you can either use optimizers/regularizers/loss/initializers with default values or you can 
# import it and customize the hyperparameters and pass it to compile

# Let's create an adam optimizer with customized learning rate =0.005
adam = tf.keras.optimizers.Adam(learning_rate=0.005)
# Let's compile the model with self_advarsarial loss of default parameters
model.compile(optimizer=adam, loss='self_adversarial')

# fit the model to data.
model.fit(dataset['train'],
             batch_size=10000,
             epochs=10,
             callbacks=[tensorboard_callback])

# the training can be visualised using the following command:
# tensorboard --logdir='./transe_train_logs' --port=8891 
# open the browser and go to the following URL: http://127.0.0.1:8891/

Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 36872.7461
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 22276.2441
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 16969.3262
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 14279.5000
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 12669.0176
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 11605.2178
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 10855.1104
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 10299.5020
Epoch 9/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 9871.7344
Epoch 10/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 9533.8877


## Model calibration

In [4]:
# calibrate on the test set
model.calibrate(dataset['test'],        # Dataset to calibrate on
                batch_size=500,         # Batch size to be used for calibration
                positive_base_rate=0.8, # prior which indicates what percentage of the dataset might be correct
                epochs=1,             # Number of epochs
                verbose=True)             

100%|███████████████████████████████████| 1/1 [00:00<00:00,  5.92it/s]


## Save model

In [5]:
from ampligraph.utils import save_model
# explictly save the model
save_model(model, 'saved_model_transE')

The path saved_model_transE already exists. This save operation will overwrite the model                 at the specified path.


## Restore model

In [6]:
from ampligraph.utils import restore_model

# restore saved models or checkpoints
model = restore_model('saved_model_transE')

Saved model does not include a db file. Skipping.


## Single prediction explanation

In [7]:
from ampligraph.explanations import ExamplE

In [8]:
explainer = ExamplE(dataset, model)

In [9]:
target_triple = dataset['test'][35]

In [10]:
print(target_triple)

['/m/02ghq' '/people/person/profession' '/m/01d_h8']


In [11]:
explanation = explainer.predict_explain(target_triple)

/Users/adri/Work/Evergreen-Projects/venv_ampli/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/adri/Work/Evergreen-Projects/venv_ampli/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/adri/Work/Evergreen-Projects/venv_ampli/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/adri/Work/Evergreen-Projects/venv_ampli/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/adri/Work/Evergreen-Projects/venv_ampli/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/adri/Work/Evergreen-Projects/venv_ampli/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered 

In [12]:
import json

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)

In [13]:
with open('explanation.json','w') as f:
    f.write(json.dumps(explanation, cls=NpEncoder))

In [14]:
with open('explanation.json') as f:
    explanation = json.loads(f.read())

In [15]:
print(explanation.keys())

dict_keys(['prediction', 'probability', 'examples', 'resembles', 'prototype', 'query_triple', 'N-hood'])


### List past examples similr to target triple

In [16]:
explanation['resembles']

[['/m/01d_h8', 'resembles', '/m/01d_h8'],
 ['/m/02ghq', 'resembles', '/m/01q4qv'],
 ['/m/02ghq', 'resembles', '/m/02lhm2']]

In [17]:
from ampligraph.datasets import load_fb15k_237
X = load_fb15k_237(return_mapper=True)
print("\n".join([f"{x[0]} resembles {x[2]}" for x in np.vectorize(X['mapper'].get)(explanation['resembles'])]))

Film Producer resembles Film Producer
Dean Koontz resembles Costa-Gavras
Dean Koontz resembles Cedric the Entertainer


In [18]:
print("\n".join([f"{x[0]} {explanation['examples'][i][1]} {x[2]}" for i, x in enumerate(np.vectorize(X['mapper'].get)(explanation['examples']))]))

Costa-Gavras /people/person/profession Film Producer
Cedric the Entertainer /people/person/profession Film Producer


## Explanations visualization

For visualizations to work you need to install extra packages pygraphviz, graphviz, pydot:

pip install pygraphviz graphviz pydot

In [20]:
explainer.visualize(explanation)

In [21]:
# Translate ids into human readable labels
explanation_translated = {'prediction': explanation['prediction'], 'probability': explanation['probability'], 'query_triple': [X['mapper'].get(explanation['query_triple'][0]), explanation['query_triple'][1], X['mapper'].get(explanation['query_triple'][2])]}
for k in ['examples', 'resembles', 'prototype', 'N-hood']:
    explanation_translated[k] = [[x[0], explanation[k][i][1], x[2]] for i, x in enumerate(np.vectorize(X['mapper'].get)(explanation[k]))]

In [22]:
explainer.visualize(explanation_translated)

In [23]:
explanation.keys()

dict_keys(['prediction', 'probability', 'examples', 'resembles', 'prototype', 'query_triple', 'N-hood'])

## Batch explanations with scores

In [24]:
explanations = explainer.batch_explain([target_triple], score=True)

WARNING - Examples shape: (1, 2, 3), (1,), (1,), (1,) 


/Users/adri/Work/Evergreen-Projects/venv_ampli/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/adri/Work/Evergreen-Projects/venv_ampli/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/adri/Work/Evergreen-Projects/venv_ampli/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/adri/Work/Evergreen-Projects/venv_ampli/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/adri/Work/Evergreen-Projects/venv_ampli/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/adri/Work/Evergreen-Projects/venv_ampli/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered 

In [25]:
from ampligraph.datasets import load_fb15k_237
X = load_fb15k_237(return_mapper=True)
print("=============\n".join(["\n".join([f"{rank+1} {X['mapper'][e[0]]} {e[1]} {X['mapper'][e[2]]} distance:{float(e[3])}" for rank, e in enumerate(exp)]) for exp in explanations]))

1 Cedric the Entertainer /people/person/profession Film Producer distance:0.2758842905362447
2 Costa-Gavras /people/person/profession Film Producer distance:0.27661744753519696
